 # Encoder Scaling and Inverse Transformation in `TimeSeriesDataSet`

`TimeSeriesDataSet` applies feature scaling internally, but until now these
scaling parameters were not easily accessible or invertible. This tutorial
introduces the new functionality that allows you to:

- inspect the encoder scaling parameters used by the dataset  
- invert the scaling applied to continuous encoder inputs  
- reconstruct original feature values for debugging, visualization, or export  

We use a small synthetic dataset to illustrate the workflow and verify that the
inverse transformation accurately recovers the raw data.

In [1]:
import numpy as np
import pandas as pd
import torch

 # 1. Creating a TimeSeriesDataSet with mixed per‑feature scalers

Create a small synthetic dataset with noise.  
This makes it easy to verify that inverse-scaling reconstructs the original data.

The data consists in two (`n_assets`) time series of length eight (`n_times`) with four (`n_covariates`) covariates.

In [2]:
from pytorch_forecasting.data.examples import generate_ar_data

n_times = 8
n_series = 2
n_covariates = 4

data = generate_ar_data(
    seasonality=n_times / 2, timesteps=n_times, n_series=n_series, seed=42
)

for f in range(1, n_covariates + 1):
    arr = np.cos(np.arange(n_times) / 5) + np.random.normal(scale=0.1, size=n_times)

    for i in range(n_series):
        mask = data["series"] == i
        data.loc[mask[mask].index, f"covariate{f}"] = arr.copy()

# force the group_id to be categorical
data["series"] = data["series"].apply(lambda i: chr(65 + i))

data

W0519 08:32:52.104000 28024 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


,series,time_idx,value,covariate1,covariate2,covariate3,covariate4
0,A,0,0.000000,0.945562,0.998650,1.073847,1.034362
1,A,1,0.360663,0.991159,0.874295,0.997203,0.803763
2,A,2,0.743944,0.805962,1.003315,0.909496,0.953469
3,A,3,1.252783,0.862905,0.703251,0.795225,0.786827
4,A,4,1.990786,0.636643,0.717593,0.548855,0.629015
5,A,5,2.158686,0.511133,0.344335,0.468318,0.601470
6,A,6,2.888813,0.302187,0.229539,0.316294,0.465458
7,A,7,4.160546,0.355195,0.189653,0.275679,0.263095
8,B,0,0.000000,0.945562,0.998650,1.073847,1.034362
9,B,1,0.015469,0.991159,0.874295,0.997203,0.803763


Define a TimeSeriesDataSet with mixed scaling strategies:

- Two unscaled features.

- One feature using sklearn `StandardScaler`

- Two (`n_scalers`) features using `EncoderNormalizer`

This setup allows us to test whether inverse-scaling works for all cases.

The encoding length is four (`in_steps`) and the prediction length is two (`out_steps`).

In [3]:
from sklearn.preprocessing import StandardScaler
from pytorch_forecasting import TimeSeriesDataSet, EncoderNormalizer

in_steps = 4
out_steps = 2

# Define the dataset
self = TimeSeriesDataSet(
    data,
    time_idx="time_idx",
    target="value",
    group_ids=["series"],
    max_encoder_length=in_steps,
    max_prediction_length=out_steps,
    #     time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=[
        "value",
        "covariate1",
        "covariate2",
        "covariate3",
        "covariate4",
    ],
    target_normalizer=EncoderNormalizer(),
    scalers={
        # Custom per-feature scalers (new feature enhancement)
        "time_idx": None,
        "covariate1": None,
        "covariate2": StandardScaler(),
        "covariate3": EncoderNormalizer(),
        "covariate4": EncoderNormalizer(),
    },
    #     add_relative_time_idx=True,
    #     add_target_scales=True,
    #     add_encoder_length=True,
)

C:\ProgramData\miniconda3\envs\test\lib\site-packages\numpy\_core\fromnumeric.py:4062: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
C:\ProgramData\miniconda3\envs\test\lib\site-packages\numpy\_core\fromnumeric.py:4062: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


 # 2. Accessing encoder scaling parameters from the dataset

The datatset has 6 elements (`n_assets * (n_times + 1 - in_steps - out_steps)`).  
A dataset element is a dictionary of `torch.Tensor` and a pair of `torch.Tensor`.

After the dataset is built, the following attributes become available. Notice the two additional keys:

- `x_scale_idx (n_scalers,)`: mapping from feature index → scale index, and

- `x_scale (n_scalers, 2)`: tensor of per-feature `(center, scale)` values.

In [4]:
print("Dataset size:", len(self))
print()

x, (y, w) = self[0]
for k, v in x.items():
    try:
        if v.ndim != 0:
            print(k, v.shape, v.dtype)
        else:
            print(k, v)
    except AttributeError:
        if isinstance(v, list):
            print(k, [(vv.shape, vv.dtype) for vv in v])
        else:
            print(k, v)
if isinstance(y, list):
    print("y", [yy.size() for yy in y])
else:
    print("y", y.size())
print("w", w)

Dataset size: 6

x_cat torch.Size([6, 0]) torch.int64
x_cont torch.Size([6, 5]) torch.float32
encoder_length 4
decoder_length 2
encoder_target torch.Size([4]) torch.float32
encoder_time_idx_start tensor(0)
groups torch.Size([1]) torch.int64
target_scale (2,) float32
x_scale_idx torch.Size([2]) torch.int64
x_scale torch.Size([2, 2]) torch.float32
y torch.Size([2])
w None


 # 3. Inverse‑transforming a dataset item

Transform a dataset continuous encoder element into its original (unscaled) values.

In [5]:
# use the new dataset_inverse_transform() function
x_cont_inv = self.inverse_scaling(
    x["x_cont"], x["target_scale"], x["x_scale_idx"], x["x_scale"]
)

# reconstruct the original data
data_reconstructed = (
    pd.DataFrame(x_cont_inv)
    .round({"time_idx": 4})  # to correctly recover integer dtypes
    .astype(data[data.columns.intersection(self.reals)].dtypes)
)

# series_id: use the existing .inverse_transform() method
data_reconstructed["series"] = (
    self.get_transformer("series", group_id=True).inverse_transform(x["groups"]).item()
)

# 'time_idx' is not necessarily part of self.reals
if "time_idx" not in data_reconstructed:
    data_reconstructed["time_idx"] = x["encoder_time_idx_start"] + torch.arange(
        x["encoder_length"] + x["decoder_length"]
    )

data_reconstructed

,value,covariate1,covariate2,covariate3,covariate4,series,time_idx
0,0.000000,0.945562,0.998650,1.073847,1.034362,A,0
1,0.360663,0.991159,0.874295,0.997203,0.803763,A,1
2,0.743944,0.805962,1.003315,0.909496,0.953469,A,2
3,1.252783,0.862905,0.703251,0.795225,0.786827,A,3
4,1.990786,0.636643,0.717593,0.548855,0.629014,A,4
5,2.158686,0.511133,0.344335,0.468318,0.601470,A,5


 ## Validation: reconstructed values match the original data

Compare inverse-transformed values to original data.

In [6]:
# identify subset of data corresponding to the element
data_original = data.merge(
    data_reconstructed[["series", "time_idx"]].drop_duplicates(),
    on=["series", "time_idx"],
    how="inner",
)[data_reconstructed.columns]

# Compare: Use dtype-dependent rounding to avoid float32 noise
round_precision = -int(np.log10(torch.finfo(x["x_cont"].dtype).resolution)) - 1

assert (
    data_reconstructed.round(round_precision)
    .compare(data_original.round(round_precision))
    .empty
), "Data are different!"

data_original

,value,covariate1,covariate2,covariate3,covariate4,series,time_idx
0,0.000000,0.945562,0.998650,1.073847,1.034362,A,0
1,0.360663,0.991159,0.874295,0.997203,0.803763,A,1
2,0.743944,0.805962,1.003315,0.909496,0.953469,A,2
3,1.252783,0.862905,0.703251,0.795225,0.786827,A,3
4,1.990786,0.636643,0.717593,0.548855,0.629015,A,4
5,2.158686,0.511133,0.344335,0.468318,0.601470,A,5


 # 4. Accessing encoder scaling parameters from a dataloader

In [7]:
dataloader = self.to_dataloader(train=False, batch_size=3)

The dataloader has 2 elements (`len(self) / batch_size`).  
A dataloader element is a dictionary of `torch.Tensor` and a pair of `torch.Tensor`.

After the dataset is built, the following attributes become available. Notice the two additional keys:

- `encoder_scale_idx (batch_size, n_scalers,)`: mapping from feature index → scale index, and

- `encoder_scale (batch_size, n_scalers, 2)`: tensor of per-feature `(center, scale)` values.

In [8]:
print("Dataloader size:", len(dataloader))
print()

x, (y, w) = list(dataloader)[1]
for k, v in x.items():
    try:
        if v.ndim != 0:
            print(k, v.shape, v.dtype)
        else:
            print(k, v)
    except AttributeError:
        if isinstance(v, list):
            print(k, [(vv.shape, vv.dtype) for vv in v])
        else:
            print(k, v)
if isinstance(y, list):
    print("y", [yy.size() for yy in y])
else:
    print("y", y.size())
print("w", w)

Dataloader size: 2

encoder_cat torch.Size([3, 4, 0]) torch.int64
encoder_cont torch.Size([3, 4, 5]) torch.float32
encoder_target torch.Size([3, 4]) torch.float32
encoder_lengths torch.Size([3]) torch.int64
decoder_cat torch.Size([3, 2, 0]) torch.int64
decoder_cont torch.Size([3, 2, 5]) torch.float32
decoder_target torch.Size([3, 2]) torch.float32
decoder_lengths torch.Size([3]) torch.int64
decoder_time_idx torch.Size([3, 2]) torch.int64
groups torch.Size([3, 1]) torch.int64
target_scale torch.Size([3, 2]) torch.float32
encoder_scale_idx torch.Size([3, 2]) torch.int64
encoder_scale torch.Size([3, 2, 2]) torch.float32
y torch.Size([3, 2])
w None


 # 5. Inverse‑transforming a dataloader item

Transform a dataloader continuous encoder element into its original (unscaled) values.

In [9]:
# use the *same* new dataset_inverse_transform() function
encoder_cont_inv = dataloader.dataset.inverse_scaling(
    x["encoder_cont"], x["target_scale"], x["encoder_scale_idx"], x["encoder_scale"]
)

# consider one specific batch element
batch_idx = 0
encoder_cont_inv_idx = {k: v[batch_idx] for k, v in encoder_cont_inv.items()}

# reconstruct the original data
data_reconstructed = (
    pd.DataFrame(encoder_cont_inv_idx)
    .round({"time_idx": 4})  # to correctly recover integer dtypes
    .astype(data[data.columns.intersection(self.reals)].dtypes)
)

# series_id: use the existing .inverse_transform() method
data_reconstructed["series"] = (
    self.get_transformer("series", group_id=True)
    .inverse_transform(x["groups"][batch_idx])
    .item()
)

# 'time_idx' is not necessarily part of self.reals
if "time_idx" not in data_reconstructed:
    data_reconstructed["time_idx"] = (
        x["decoder_time_idx"][batch_idx, 0]
        - x["encoder_lengths"][batch_idx]
        + torch.arange(x["encoder_lengths"][batch_idx])
    )

data_reconstructed

,value,covariate1,covariate2,covariate3,covariate4,series,time_idx
0,0.000000,0.945562,0.998650,1.073847,1.034362,B,0
1,0.015469,0.991159,0.874295,0.997203,0.803763,B,1
2,0.126900,0.805962,1.003315,0.909496,0.953469,B,2
3,0.320945,0.862905,0.703251,0.795225,0.786827,B,3


 ## Validation: reconstructed values match the original data
 
Compare inverse-transformed values to original data.

In [10]:
# identify subset of data corresponding to the element
data_original = data.merge(
    data_reconstructed[["series", "time_idx"]].drop_duplicates(),
    on=["series", "time_idx"],
    how="inner",
)[data_reconstructed.columns]

# Compare: Use dtype-dependent rounding to avoid float32 noise
round_precision = -int(np.log10(torch.finfo(x["encoder_cont"].dtype).resolution)) - 1

assert (
    data_reconstructed.round(round_precision)
    .compare(data_original.round(round_precision))
    .empty
), "Data are different!"

data_original

,value,covariate1,covariate2,covariate3,covariate4,series,time_idx
0,0.000000,0.945562,0.998650,1.073847,1.034362,B,0
1,0.015469,0.991159,0.874295,0.997203,0.803763,B,1
2,0.126900,0.805962,1.003315,0.909496,0.953469,B,2
3,0.320945,0.862905,0.703251,0.795225,0.786827,B,3


 ## Summary

In this tutorial, we demonstrated how to use mixed per‑feature scalers in
`TimeSeriesDataSet` and how to recover original values using the new
`inverse_scaling` utility. By combining user‑defined sklearn scalers,
PyTorch Forecasting’s internal normalizers, and unscaled features, we created a
flexible and transparent scaling pipeline.

We inspected the encoder scaling parameters stored by the dataset, applied
inverse transformations to both dataset items and dataloader batches, and
verified that the reconstructed values match the original data. This workflow
enables clearer debugging, more interpretable model inputs, and easier export of
predictions back into the raw data space.

These tools provide a solid foundation for building models that remain both
powerful and interpretable, especially in settings where understanding the
scaled inputs is as important as the predictions themselves.